# Narrador + GPT-SoVITS online (Google Colab, GPU grátis)

Este notebook sobe o **GPT-SoVITS** (servidor `api_v2`) e o **Narrador** na GPU gratuita do Colab,
com uma **interface web** acessível pelo navegador (link público temporário). Nela você prepara a voz,
testa e gera o MP4 narrado.

> O GPT-SoVITS oficial não tem português; este notebook aplica o **patch pt-BR do Narrador**. Sem treino (zero-shot) o
> português sai com sotaque e erros. Para qualidade real, use o notebook `narrador_gptsovits_ptbr_treino.ipynb`, que
> treina a sua voz em pt-BR. O `chatterbox-ptbr` também está disponível na interface.

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)*.

In [ ]:
#@title 1. Instalar o GPT-SoVITS (6-10 min)
%cd /content
!git clone -q --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS
!apt-get -qq install -y ffmpeg libreoffice-impress > /dev/null
!pip install -q -r requirements.txt 2>&1 | tail -2
!pip install -q huggingface_hub
import os
from huggingface_hub import snapshot_download
snapshot_download("lj1995/GPT-SoVITS", local_dir="GPT_SoVITS/pretrained_models",
                  allow_patterns=["s1v3.ckpt", "v2Pro/*", "sv/*", "chinese-roberta-wwm-ext-large/*", "chinese-hubert-base/*"])
snapshot_download("lj1995/GPT-SoVITS", local_dir="GPT_SoVITS/text", allow_patterns=["G2PWModel/*"], repo_type="model") if False else None
import nltk; nltk.download("averaged_perceptron_tagger_eng", quiet=True); nltk.download("cmudict", quiet=True)
!pip install -q torchcodec resampy
!git clone -q https://github.com/mscpatricio/OBRAS-SAUDE.git /content/repo
!cp /content/repo/narrador/narrador/gptsovits_ptbr/*.py /content/GPT-SoVITS/ && python aplicar_patch.py /content/GPT-SoVITS
print("GPT-SoVITS pronto, com suporte a pt-BR")

In [ ]:
#@title 2. Configurar v2Pro e iniciar o servidor api_v2 (porta 9880)
import yaml, subprocess, time, requests
cfg_path = "/content/GPT-SoVITS/GPT_SoVITS/configs/tts_infer.yaml"
cfg = yaml.safe_load(open(cfg_path))
cfg["custom"] = {
    "version": "v2Pro", "device": "cuda", "is_half": True,
    "t2s_weights_path": "GPT_SoVITS/pretrained_models/s1v3.ckpt",
    "vits_weights_path": "GPT_SoVITS/pretrained_models/v2Pro/s2Gv2Pro.pth",
    "bert_base_path": "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large",
    "cnhuhbert_base_path": "GPT_SoVITS/pretrained_models/chinese-hubert-base",
}
yaml.safe_dump(cfg, open(cfg_path, "w"))
log = open("/content/api_v2.log", "w")
proc = subprocess.Popen(["python", "api_v2.py", "-a", "127.0.0.1", "-p", "9880", "-c", cfg_path],
                        cwd="/content/GPT-SoVITS", stdout=log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:9880/control", params={"command": ""}, timeout=2); break
    except Exception:
        time.sleep(2)
print("Servidor GPT-SoVITS no ar em http://127.0.0.1:9880 (log em /content/api_v2.log)")

In [ ]:
#@title 3. Instalar o Narrador (+ Chatterbox pt-BR para português)
!git clone -q https://github.com/mscpatricio/OBRAS-SAUDE.git /content/repo
%pip install -q /content/repo/narrador[web,online] chatterbox-tts 2>&1 | tail -1
print("Narrador instalado")

In [ ]:
#@title 4. Abrir a interface web (link público)
#@markdown Na interface: aba **1. Voz** (envie 8-15 s da sua voz), **2. Teste**, **3. Vídeo**.
#@markdown Para o GPT-SoVITS escolha o motor `gpt-sovits` e nas opções o patch pt-BR já está aplicado: use `text_lang=pt`.
#@markdown Para português escolha `chatterbox-ptbr`.
import os
os.environ.setdefault("GPT_SOVITS_URL", "http://127.0.0.1:9880")
!cd /content && narrador web --compartilhar --host 0.0.0.0 --saida /content/saida --motor gpt-sovits -o text_lang=pt

### Voz idêntica com o GPT-SoVITS (fine-tune, opcional)

O zero-shot usa só 3-10 s de referência. Para fidelidade máxima, o GPT-SoVITS permite treinar com **1 a 5 minutos** da sua voz:

1. Interrompa a célula 4 e rode: `!cd /content/GPT-SoVITS && is_share=True python webui.py`
2. Na WebUI: aba *0* (fatiar o áudio e transcrever com ASR), *1A* (formatar o dataset), *1B* (treinar SoVITS por ~8 épocas e GPT por ~15 épocas).
3. Os pesos saem em `SoVITS_weights_v2Pro/` e `GPT_weights_v2Pro/`. Rode de novo a célula 4 acrescentando
   `-o gpt_weights=/content/GPT-SoVITS/GPT_weights_v2Pro/SEU.ckpt -o sovits_weights=/content/GPT-SoVITS/SoVITS_weights_v2Pro/SEU.pth`.

O treino vale apenas para os idiomas do GPT-SoVITS; com texto em português ele continua inutilizável.

In [ ]:
#@title 5. Baixar o último vídeo gerado
from google.colab import files
import glob, os
vids = sorted(glob.glob("/content/saida/*/*.mp4"), key=os.path.getmtime)
files.download(vids[-1]); files.download(vids[-1].replace(".mp4", ".srt"))